In [ ]:
pip install diffusers transformers accelerate peft datasets torchvision kagglehub

In [ ]:
import kagglehub
import os

path = kagglehub.dataset_download("wenewone/cub2002011")
print("Dataset path:", path)

100%|██████████| 1.49G/1.49G [00:19<00:00, 82.3MB/s]

Extracting files...


Dataset path: /root/.cache/kagglehub/datasets/wenewone/cub2002011/versions/7


In [ ]:
from PIL import Image
import os

image_folder = os.path.join(path, "CUB_200_2011/images")

data = []

for root, dirs, files in os.walk(image_folder):
    for file in files:
        if file.endswith(".jpg"):
            img_path = os.path.join(root, file)

            # Create simple caption
            caption = "a photo of a bird"

            data.append({"image": img_path, "text": caption})

print("Total images:", len(data))

Total images: 11796


In [ ]:
from datasets import Dataset

dataset = Dataset.from_list(data)
dataset = dataset.train_test_split(test_size=0.1)

In [ ]:
from diffusers import StableDiffusionPipeline
import torch

model_id = "runwayml/stable-diffusion-v1-5"

pipe = StableDiffusionPipeline.from_pretrained(
    model_id,
    torch_dtype=torch.float16
).to("cuda")

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model_index.json:   0%|          | 0.00/541 [00:00<?, ?B/s]

Fetching 15 files:   0%|          | 0/15 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/396 [00:00<?, ?it/s]

StableDiffusionSafetyChecker LOAD REPORT from: /root/.cache/huggingface/hub/models--runwayml--stable-diffusion-v1-5/snapshots/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/safety_checker
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
vision_model.vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: /root/.cache/huggingface/hub/models--runwayml--stable-diffusion-v1-5/snapshots/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


AssertionError: Torch not compiled with CUDA enabled

In [ ]:
from peft import LoraConfig

lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["to_k", "to_q", "to_v", "to_out.0"],
    lora_dropout=0.1,
)

In [ ]:
from torch.utils.data import DataLoader
from torchvision import transforms

transform = transforms.Compose([
    transforms.Resize((512, 512)),
    transforms.ToTensor()
])

def preprocess(example):
    image = Image.open(example["image"]).convert("RGB")
    example["pixel_values"] = transform(image)
    return example

dataset = dataset.map(preprocess, batched=True, batch_size=16)